## **UCUCI Bank Loan Repayment Analysis**

This notebook looks at UCUCI Bank's loan data to understand repayment patterns and current repayment risk. Closed loans are used to study past repayment behaviour. Active loans are used to check current exposure and collection priority.


# **Business Problem Overview**

UCUCI Bank wants to improve repayment performance and reduce default risk across its loan book. The analysis focuses on borrower grade, sub-grade, income, DTI, interest rate, term, purpose, repayment amount, and current loan status.

Main question: **Which borrower segments repay better, and where should recovery effort be focused for active loans?**


### **Objective**

The objective is to compare repayment across key borrower and loan segments, identify active-loan risk, and suggest practical recovery actions.


### **Business Impact**

The analysis can help the bank identify weaker repayment segments, monitor risky active loans earlier, and focus collection effort where outstanding exposure is high.


### **Metric Framework and Supporting KPIs**

Main metric: **Loan Repayment Rate (LRR)**

**LRR = (Total Amount Repaid / Total Amount Funded) x 100**

LRR is mainly interpreted for **closed loans** in this notebook, because active loans are still being repaid and their final repayment outcome is not complete. LRR can also be above 100% because total payment includes principal plus interest, so good fully-paid loans can repay more than the original funded amount.

Supporting KPIs used:

- **Charged-off rate**: Share of loans moved to Charged Off.
- **Bad-loan rate**: Share of loans that are Charged Off or Default.
- **Average interest rate**: Checks grade-based pricing.
- **Practical recovery rate**: recoveries / funded amount for charged-off loans.
- **Active delinquency rate**: Share of active loans in grace or late status.
- **Outstanding principal exposure**: Current money still at risk in active loans.


# **Dataset Overview**

- Dataset: UCUCI Bank Loan Dataset
- Rows: 887,379
- Columns: 57
- Main target field: loan_status
- Closed statuses used: Fully Paid, Charged Off, Default
- Active statuses used: Current, In Grace Period, Late (16-30 days), Late (31-120 days)

Important columns used include loan amount, funded amount, term, interest rate, installment, grade, sub-grade, income, DTI, purpose, total payment, outstanding principal, and recoveries.


# **Analysis & Visualisation**


## ***1. Importing and Loading Data***

### Importing Necessary Libraries


In [ ]:
# Install gdown to download the dataset from Google Drive
!pip install gdown -q

import gdown              # To download the file from Google Drive
import pandas as pd       # To work with tables and dataframes
import numpy as np        # To work with numerical calculations
import matplotlib.pyplot as plt  # To create basic charts
import seaborn as sns     # To create better looking charts like heatmaps


In [ ]:
# Loading the dataset from Google Drive
file_id = "1BS8azi6nWEXm0Zvo3nD7t3sWaZWt1Ny8"
download_url = f"https://drive.google.com/uc?id={file_id}"
output_file = "UCUCI_dataset.csv"

gdown.download(download_url, output_file, quiet=False)
data = pd.read_csv(output_file)


In [ ]:
# Viewing the first few rows
data.head()


In [ ]:
# Checking shape and duplicate ids
rows, columns = data.shape
print(f"Dataset has {rows:,} rows and {columns} columns.")

print("Duplicate loan ids:", data["id"].duplicated().sum())
print("Duplicate member ids:", data["member_id"].duplicated().sum())


In [ ]:
# Checking missing values
missing = data.isnull().sum().sort_values(ascending=False).to_frame("missing_count")
missing["missing_percent"] = (missing["missing_count"] / len(data)) * 100
missing.head(20)


### Summary of Initial Dataset Observations

- Dataset size: **887,379 rows and 57 columns**.
- No duplicate loan IDs or member IDs were found.
- Some columns have high missing values, but they are not central to this repayment analysis.
- Main fields used: loan_status, grade, term, purpose, dti, int_rate, funded_amnt, total_pymnt, and out_prncp.


## ***Data Understanding***

### Loan Status Distribution


In [ ]:
data["loan_status"].value_counts()


The dataset has both closed and active loans. They are separated because they answer two different business questions:

- **Closed loans** show how repayment has already behaved.
- **Active loans** show where the bank currently has exposure and possible default risk.

### Important Categorical Variables


In [ ]:
print("Grade distribution:")
print(data["grade"].value_counts().sort_index())

print("\nTerm distribution:")
print(data["term"].value_counts())

print("\nTop loan purposes:")
print(data["purpose"].value_counts().head(15))


In [ ]:
# Summary statistics for important numerical columns
data[["loan_amnt", "annual_inc", "dti", "int_rate", "installment", "funded_amnt", "total_pymnt", "out_prncp"]].describe(
    percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
)


### Data Quality Notes

A few DTI values are above 100, so DTI charts use records with DTI <= 100. Very high income values are noted but not removed because the analysis is mainly segment-based.


In [ ]:
print("Loans with DTI greater than 100:", (data["dti"] > 100).sum())
print("Loans with annual income greater than 10,00,000:", (data["annual_inc"] > 1000000).sum())


### Exposure Amounts in Crores

Large funded and outstanding amounts are also shown in crores where needed.

**Amount in crores = Amount / 1,00,00,000**


## ***Data Preparation***

Two working datasets are created:

1. completed: Loans that reached a final status.
2. active_loans: Loans that are still ongoing.

A small function is used for LRR so the formula stays clear.


In [ ]:
def calculate_lrr(total_repaid, total_funded):
    # Formula: LRR = total repaid / total funded * 100
    lrr_value = (total_repaid / total_funded) * 100
    return lrr_value

# Creating the main project metric
data["lrr"] = calculate_lrr(data["total_pymnt"], data["funded_amnt"])

completed_status = ["Fully Paid", "Charged Off", "Default"]
active_status = ["Current", "In Grace Period", "Late (16-30 days)", "Late (31-120 days)"]
delinquent_status = ["In Grace Period", "Late (16-30 days)", "Late (31-120 days)"]

# Separating completed and active loans
completed_filter = data["loan_status"].isin(completed_status)
active_filter = data["loan_status"].isin(active_status)

completed = data[completed_filter].copy()
active_loans = data[active_filter].copy()

completed_repaid = completed["total_pymnt"].sum()
completed_funded = completed["funded_amnt"].sum()
overall_completed_lrr = calculate_lrr(completed_repaid, completed_funded)

active_delinquent_loans = active_loans[active_loans["loan_status"].isin(delinquent_status)]
active_delinquency_rate = (len(active_delinquent_loans) / len(active_loans)) * 100

print("Completed loans:", completed.shape[0])
print("Active loans:", active_loans.shape[0])
print("Overall completed-loan LRR:", round(overall_completed_lrr, 2), "%")
print("Active-loan delinquency rate:", round(active_delinquency_rate, 2), "%")


In [ ]:
charged_off = data[data["loan_status"] == "Charged Off"].copy()

charged_off_outstanding = charged_off["out_prncp"].sum()
charged_off_recoveries = charged_off["recoveries"].sum()

print("Charged-off outstanding principal:", charged_off_outstanding)
print("Charged-off recoveries:", round(charged_off_recoveries, 2))

if charged_off_outstanding > 0:
    recovery_rate_outstanding = (charged_off_recoveries / charged_off_outstanding) * 100
    print("Recovery rate using recoveries / out_prncp:", round(recovery_rate_outstanding, 2), "%")
else:
    print("Recovery rate using recoveries / out_prncp cannot be calculated because out_prncp is 0 for charged-off closed loans.")

recovery_rate_funded = (charged_off_recoveries / charged_off["funded_amnt"].sum()) * 100
print("Practical recovery rate using recoveries / funded_amnt:", round(recovery_rate_funded, 2), "%")


### Recovery Rate Note

For charged-off loans, out_prncp is zero because these loans have already closed their outstanding principal position. So the practical recovery metric used here is:

**Practical Recovery Rate = recoveries / funded_amnt**

### Supporting KPI Overview

The main KPIs are calculated once before moving into segment-level analysis.


In [ ]:
total_loans_issued = len(data)
charged_off_loans = len(data[data["loan_status"] == "Charged Off"])
default_rate = (charged_off_loans / total_loans_issued) * 100

bad_loan_count = len(data[data["loan_status"].isin(["Charged Off", "Default"])])
bad_loan_rate = (bad_loan_count / total_loans_issued) * 100
avg_interest_overall = data["int_rate"].mean()

print("Total loans issued:", total_loans_issued)
print("Charged-off loans:", charged_off_loans)
print("Charged-off rate:", round(default_rate, 2), "%")
print("Charged-off + Default rate:", round(bad_loan_rate, 2), "%")
print("Average interest rate overall:", round(avg_interest_overall, 2), "%")


Charged-off rate is the main default-risk KPI. Charged Off + Default is shown as a wider risk view.


# **Closed Loan Analysis**


## ***Hypothesis 1: Lower credit grades have lower repayment rates***


In [ ]:
grade_group = completed.groupby("grade")

grade_summary = pd.DataFrame()
grade_summary["loans"] = grade_group["id"].count()
grade_summary["funded"] = grade_group["funded_amnt"].sum()
grade_summary["repaid"] = grade_group["total_pymnt"].sum()
grade_summary["lrr"] = calculate_lrr(grade_summary["repaid"], grade_summary["funded"])

charged_off_only = completed[completed["loan_status"] == "Charged Off"]
bad_loans_only = completed[completed["loan_status"].isin(["Charged Off", "Default"])]

charged_off_by_grade = charged_off_only.groupby("grade")["id"].count()
bad_loans_by_grade = bad_loans_only.groupby("grade")["id"].count()

grade_summary["charged_off_count"] = charged_off_by_grade
grade_summary["bad_loan_count"] = bad_loans_by_grade

grade_summary["charged_off_count"] = grade_summary["charged_off_count"].fillna(0)
grade_summary["bad_loan_count"] = grade_summary["bad_loan_count"].fillna(0)

grade_summary["charged_off_rate"] = (grade_summary["charged_off_count"] / grade_summary["loans"]) * 100
grade_summary["bad_loan_rate"] = (grade_summary["bad_loan_count"] / grade_summary["loans"]) * 100
grade_summary["funded_crore"] = grade_summary["funded"] / 10000000

display(grade_summary.round(2))

plt.figure(figsize=(8, 5))
plt.bar(grade_summary.index, grade_summary["lrr"])
plt.title("Loan Repayment Rate by Grade")
plt.xlabel("Grade")
plt.ylabel("LRR (%)")
plt.show()


This section checks whether borrower grade is linked with repayment quality. Grade is expected to represent credit risk, so lower grades should ideally show weaker repayment if the grading system is working properly.

The result supports this pattern. LRR falls from grade A/B toward grade G. Grades A and B cross 100% because total_pymnt includes principal plus interest, which is expected for good closed loans. The lower grades show weaker repayment, so they need closer tracking from the start.


## ***Supporting KPI: Average Interest Rate by Grade and Sub-grade***


In [ ]:
grade_group = completed.groupby("grade")

avg_interest_grade = pd.DataFrame()
avg_interest_grade["loans"] = grade_group["id"].count()
avg_interest_grade["avg_interest_rate"] = grade_group["int_rate"].mean()
avg_interest_grade["avg_lrr"] = grade_group["lrr"].mean()

charged_off_only = completed[completed["loan_status"] == "Charged Off"]
charged_off_count = charged_off_only.groupby("grade")["id"].count()

avg_interest_grade["charged_off_count"] = charged_off_count
avg_interest_grade["charged_off_count"] = avg_interest_grade["charged_off_count"].fillna(0)
avg_interest_grade["charged_off_rate"] = (avg_interest_grade["charged_off_count"] / avg_interest_grade["loans"]) * 100

display(avg_interest_grade.round(2))

plt.figure(figsize=(8, 5))
plt.plot(avg_interest_grade.index, avg_interest_grade["avg_interest_rate"], marker="o")
plt.title("Average Interest Rate by Grade")
plt.xlabel("Grade")
plt.ylabel("Average Interest Rate (%)")
plt.show()


In [ ]:
subgrade_group = completed.groupby("sub_grade")

avg_interest_subgrade = pd.DataFrame()
avg_interest_subgrade["loans"] = subgrade_group["id"].count()
avg_interest_subgrade["avg_interest_rate"] = subgrade_group["int_rate"].mean()
avg_interest_subgrade["avg_lrr"] = subgrade_group["lrr"].mean()

avg_interest_subgrade = avg_interest_subgrade.sort_index()
avg_interest_subgrade.head(10).round(2)


This section checks whether the bank is charging higher interest rates to riskier grades and whether that pricing is enough to cover risk.

Average interest rate rises as grade becomes riskier, which means risk-based pricing is visible in the data. However, lower grades still show weaker repayment and higher charged-off rates. This means pricing helps, but it does not fully solve repayment risk by itself.


## ***Hypothesis 2: Higher interest rate loans have lower repayment rates***


In [ ]:
# Creating interest rate bands first
completed["rate_band"] = pd.cut(
    completed["int_rate"],
    bins=[0, 10, 15, 20, 25, 30],
    labels=["0-10", "10-15", "15-20", "20-25", "25-30"],
    include_lowest=True
)

rate_group = completed.groupby("rate_band", observed=False)

rate_bands_lrr = pd.DataFrame()
rate_bands_lrr["loans"] = rate_group["id"].count()
rate_bands_lrr["funded"] = rate_group["funded_amnt"].sum()
rate_bands_lrr["repaid"] = rate_group["total_pymnt"].sum()
rate_bands_lrr["lrr"] = calculate_lrr(rate_bands_lrr["repaid"], rate_bands_lrr["funded"])

display(rate_bands_lrr.round(2))

plt.figure(figsize=(8, 5))
plt.plot(rate_bands_lrr.index.astype(str), rate_bands_lrr["lrr"], marker="o")
plt.title("LRR by Interest Rate Band")
plt.xlabel("Interest Rate Band (%)")
plt.ylabel("LRR (%)")
plt.show()


This hypothesis checks whether higher interest rate loans have lower repayment performance. Higher interest can increase the borrower's repayment burden, so repayment may become weaker as interest rate increases.

The result shows that LRR decreases as interest rate bands increase. This suggests that high-interest borrowers should be monitored earlier, especially before they enter late payment buckets.


## ***Hypothesis 3: Higher DTI borrowers have lower repayment rates***


In [ ]:
# Keeping only reasonable DTI values for this chart
completed_dti = completed[completed["dti"] <= 100].copy()

completed_dti["dti_band"] = pd.cut(
    completed_dti["dti"],
    bins=[0, 10, 20, 30, 40, 100],
    include_lowest=True
)

dti_group = completed_dti.groupby("dti_band", observed=False)

dti_band_lrr = pd.DataFrame()
dti_band_lrr["loans"] = dti_group["id"].count()
dti_band_lrr["funded"] = dti_group["funded_amnt"].sum()
dti_band_lrr["repaid"] = dti_group["total_pymnt"].sum()
dti_band_lrr["lrr"] = calculate_lrr(dti_band_lrr["repaid"], dti_band_lrr["funded"])

display(dti_band_lrr.round(2))

plt.figure(figsize=(8, 5))
plt.plot(dti_band_lrr.index.astype(str), dti_band_lrr["lrr"], marker="o")
plt.title("LRR by DTI Band")
plt.xlabel("DTI Band")
plt.ylabel("LRR (%)")
plt.xticks(rotation=30)
plt.show()


This hypothesis checks whether borrowers with higher debt-to-income ratio have lower repayment performance. DTI is useful because it shows how much of a borrower's income is already tied to debt obligations.

The result shows that LRR declines as DTI increases. Borrowers with high DTI may need stricter checks, lower loan limits, or earlier follow-up because their repayment cushion is smaller.


## ***Hypothesis 3B: Annual income is related to repayment performance***


In [ ]:
completed_income = completed[completed["annual_inc"].notna()].copy()

completed_income["income_band"] = pd.cut(
    completed_income["annual_inc"],
    bins=[0, 30000, 60000, 100000, 150000, 1000000, completed_income["annual_inc"].max()],
    labels=["0-30K", "30K-60K", "60K-100K", "100K-150K", "150K-1M", "1M+"],
    include_lowest=True
)

income_group = completed_income.groupby("income_band", observed=False)

income_summary = pd.DataFrame()
income_summary["loans"] = income_group["id"].count()
income_summary["funded"] = income_group["funded_amnt"].sum()
income_summary["repaid"] = income_group["total_pymnt"].sum()
income_summary["lrr"] = calculate_lrr(income_summary["repaid"], income_summary["funded"])

display(income_summary.round(2))

plt.figure(figsize=(8, 5))
plt.plot(income_summary.index.astype(str), income_summary["lrr"], marker="o")
plt.title("LRR by Annual Income Band")
plt.xlabel("Annual Income Band")
plt.ylabel("LRR (%)")
plt.xticks(rotation=30)
plt.show()


This section checks whether annual income alone explains repayment behaviour. Income is important, but it does not always show the full repayment capacity of a borrower.

The result suggests that income should not be read alone. A borrower can have high income and still be risky if DTI or installment burden is high. Income is more useful when combined with DTI, installment amount, term, and grade.


## ***Hypothesis 4: 60-month loans repay worse than 36-month loans***


In [ ]:
term_group = completed.groupby("term")

term_summary = pd.DataFrame()
term_summary["loans"] = term_group["id"].count()
term_summary["funded"] = term_group["funded_amnt"].sum()
term_summary["repaid"] = term_group["total_pymnt"].sum()
term_summary["lrr"] = calculate_lrr(term_summary["repaid"], term_summary["funded"])

display(term_summary.round(2))

plt.figure(figsize=(8, 3))
plt.barh(term_summary.index, term_summary["lrr"])
plt.title("LRR by Loan Term")
plt.xlabel("LRR (%)")
plt.ylabel("Term")
plt.show()


This hypothesis checks whether longer loan tenure is linked with weaker repayment. A 60-month loan keeps the borrower in repayment for a longer period, so the chance of financial stress can increase.

The result shows that 60-month loans have weaker repayment than 36-month loans. Longer tenure seems to add repayment risk, especially when combined with lower grades or higher interest rates.


## ***Hypothesis 4B: Higher installment amounts can affect repayment performance***


In [ ]:
completed["installment_band"] = pd.cut(
    completed["installment"],
    bins=[0, 200, 400, 600, 800, 1000, completed["installment"].max()],
    labels=["0-200", "200-400", "400-600", "600-800", "800-1000", "1000+"],
    include_lowest=True
)

installment_group = completed.groupby("installment_band", observed=False)

installment_summary = pd.DataFrame()
installment_summary["loans"] = installment_group["id"].count()
installment_summary["funded"] = installment_group["funded_amnt"].sum()
installment_summary["repaid"] = installment_group["total_pymnt"].sum()
installment_summary["lrr"] = calculate_lrr(installment_summary["repaid"], installment_summary["funded"])

display(installment_summary.round(2))

plt.figure(figsize=(8, 5))
plt.plot(installment_summary.index.astype(str), installment_summary["lrr"], marker="o")
plt.title("LRR by Monthly Installment Band")
plt.xlabel("Monthly Installment Band")
plt.ylabel("LRR (%)")
plt.xticks(rotation=30)
plt.show()


This section checks whether monthly installment amount affects repayment performance. Installment amount is important because it reflects the monthly burden on the borrower.

The result shows that installment should be checked along with income and DTI. A high installment may be manageable for one borrower but risky for another, depending on income, debt burden, grade, and term.


## ***Hypothesis 5: Loan purpose affects repayment behaviour***


In [ ]:
purpose_group = completed.groupby("purpose")

purpose_summary = pd.DataFrame()
purpose_summary["loans"] = purpose_group["id"].count()
purpose_summary["funded"] = purpose_group["funded_amnt"].sum()
purpose_summary["repaid"] = purpose_group["total_pymnt"].sum()
purpose_summary["lrr"] = calculate_lrr(purpose_summary["repaid"], purpose_summary["funded"])

display(purpose_summary.sort_values("lrr").round(2))

purpose_lrr = purpose_summary.sort_values("lrr")

plt.figure(figsize=(10, 6))
plt.barh(purpose_lrr.index, purpose_lrr["lrr"])
plt.title("LRR by Loan Purpose")
plt.xlabel("LRR (%)")
plt.ylabel("Purpose")
plt.show()


This hypothesis checks whether loan purpose has any relationship with repayment behaviour. Purpose matters because repayment ability may differ between personal expenses, credit card refinancing, business needs, and other loan types.

The result shows that small business loans have weaker closed-loan repayment. Loan purpose should not be used alone to approve or reject loans, but it can be treated as one useful risk signal.


## ***Hypothesis 6: Grade and term together explain repayment risk better***


In [ ]:
grade_term_group = completed.groupby(["grade", "term"])

grade_term = pd.DataFrame()
grade_term["loans"] = grade_term_group["id"].count()
grade_term["funded"] = grade_term_group["funded_amnt"].sum()
grade_term["repaid"] = grade_term_group["total_pymnt"].sum()
grade_term["lrr"] = calculate_lrr(grade_term["repaid"], grade_term["funded"])

display(grade_term.round(2))

grade_term_pivot = grade_term["lrr"].unstack()

plt.figure(figsize=(8, 5))
sns.heatmap(grade_term_pivot, annot=True, fmt=".1f", cmap="RdYlGn")
plt.title("LRR by Grade and Term")
plt.xlabel("Term")
plt.ylabel("Grade")
plt.show()


This hypothesis checks grade and term together instead of looking at them separately. This is useful because risk can become clearer when two variables are combined.

The heatmap shows that lower grades with 60-month terms have weaker repayment. This means long-tenure loans for risky grades need extra caution, stronger affordability checks, or closer monitoring after disbursement.


## ***Sub-grade Risk Pattern***


In [ ]:
subgrade_group = completed.groupby("sub_grade")
subgrade_total = subgrade_group["id"].count()

charged_off_only = completed[completed["loan_status"] == "Charged Off"]
subgrade_charged_off_count = charged_off_only.groupby("sub_grade")["id"].count()

subgrade_charged_off = pd.DataFrame()
subgrade_charged_off["total_loans"] = subgrade_total
subgrade_charged_off["charged_off_loans"] = subgrade_charged_off_count
subgrade_charged_off["charged_off_loans"] = subgrade_charged_off["charged_off_loans"].fillna(0)
subgrade_charged_off["charged_off_rate"] = (subgrade_charged_off["charged_off_loans"] / subgrade_charged_off["total_loans"]) * 100

display(subgrade_charged_off["charged_off_rate"].round(2))

plt.figure(figsize=(15, 5))
subgrade_charged_off["charged_off_rate"].plot(kind="bar")
plt.title("Charged-off Rate by Sub-grade")
plt.xlabel("Sub-grade")
plt.ylabel("Charged-off Rate (%)")
plt.show()


This section goes one level deeper than grade and checks sub-grade risk. Sub-grades are useful because two borrowers in the same broad grade may still have different risk levels.

The result shows that sub-grades give a more detailed risk view than broad grades. This can help the bank prioritise monitoring more precisely, especially within lower-grade borrowers.


## ***Loan Age Pattern***

This section checks **when** risk appears in the loan lifecycle. Since exact charge-off date is not available, the difference between issue_d and last_pymnt_d is used as a simple proxy for loan age.


In [ ]:
loan_age_data = completed.copy()

# Converting date columns to datetime
loan_age_data["issue_date"] = pd.to_datetime(loan_age_data["issue_d"], format="%b-%Y", errors="coerce")
loan_age_data["last_payment_date"] = pd.to_datetime(loan_age_data["last_pymnt_d"], format="%b-%Y", errors="coerce")

# Calculating approximate loan age in months
issue_year = loan_age_data["issue_date"].dt.year
issue_month = loan_age_data["issue_date"].dt.month
last_payment_year = loan_age_data["last_payment_date"].dt.year
last_payment_month = loan_age_data["last_payment_date"].dt.month

loan_age_data["loan_age_months"] = ((last_payment_year - issue_year) * 12) + (last_payment_month - issue_month)

# Removing blank or negative values
loan_age_data = loan_age_data[loan_age_data["loan_age_months"].notna()].copy()
loan_age_data = loan_age_data[loan_age_data["loan_age_months"] >= 0].copy()

loan_age_data["loan_age_bucket"] = pd.cut(
    loan_age_data["loan_age_months"],
    bins=[0, 6, 12, 24, 36, 48, 60, 120],
    labels=["0-6", "6-12", "12-24", "24-36", "36-48", "48-60", "60+"],
    include_lowest=True
)

age_group = loan_age_data.groupby("loan_age_bucket", observed=False)

age_default_summary = pd.DataFrame()
age_default_summary["loans"] = age_group["id"].count()

charged_off_age = loan_age_data[loan_age_data["loan_status"] == "Charged Off"]
charged_off_by_age = charged_off_age.groupby("loan_age_bucket", observed=False)["id"].count()

age_default_summary["charged_off_count"] = charged_off_by_age
age_default_summary["charged_off_count"] = age_default_summary["charged_off_count"].fillna(0)
age_default_summary["charged_off_rate"] = (age_default_summary["charged_off_count"] / age_default_summary["loans"]) * 100
age_default_summary["avg_lrr"] = age_group["lrr"].mean()

display(age_default_summary.round(2))

plt.figure(figsize=(8, 5))
plt.plot(age_default_summary.index.astype(str), age_default_summary["charged_off_rate"], marker="o")
plt.title("Charged-off Rate by Loan Age Bucket")
plt.xlabel("Loan Age Bucket in Months")
plt.ylabel("Charged-off Rate (%)")
plt.show()


In [ ]:
highest_age_bucket = age_default_summary["charged_off_rate"].idxmax()
highest_age_rate = age_default_summary.loc[highest_age_bucket, "charged_off_rate"]

print(
    "In the current output, the", highest_age_bucket,
    "month bucket has the highest charged-off rate at",
    round(highest_age_rate, 2), "%"
)

print("This suggests that the bank should focus intervention around this part of the loan lifecycle.")


This section checks when charged-off risk appears in the loan lifecycle. Since exact charge-off date is not available, loan age is estimated using issue date and last payment date.

The code identifies the loan-age bucket with the highest charged-off rate. That bucket is useful because it shows when intervention may be most important, such as early follow-up or mid-tenure repayment support.


# **Active Loan Analysis**


## ***Active Loan Delinquency Overview***


In [ ]:
active_delinquent_loans = active_loans[active_loans["loan_status"].isin(delinquent_status)]
delinquency_rate = (len(active_delinquent_loans) / len(active_loans)) * 100
print("Active-loan delinquency rate:", round(delinquency_rate, 2), "%")


This section checks the overall delinquency level in active loans. Active delinquency is important because these loans have not fully defaulted yet, so the bank still has time to act.

The active-loan delinquency rate is around **3.25%** in the existing output. This gives a starting point for understanding how much of the active portfolio needs immediate monitoring.


## ***Hypothesis 7: Lower grades have higher active-loan delinquency***


In [ ]:
grade_active_total = active_loans.groupby("grade")["id"].count()

delinquent_active = active_loans[active_loans["loan_status"].isin(delinquent_status)]
grade_active_delinquent = delinquent_active.groupby("grade")["id"].count()

grade_delinquency = pd.DataFrame()
grade_delinquency["active_loans"] = grade_active_total
grade_delinquency["delinquent_loans"] = grade_active_delinquent
grade_delinquency["delinquent_loans"] = grade_delinquency["delinquent_loans"].fillna(0)
grade_delinquency["delinquency_rate"] = (grade_delinquency["delinquent_loans"] / grade_delinquency["active_loans"]) * 100

display(grade_delinquency["delinquency_rate"].round(2))

plt.figure(figsize=(8, 5))
plt.plot(grade_delinquency.index, grade_delinquency["delinquency_rate"], marker="o")
plt.title("Active Loan Delinquency Rate by Grade")
plt.xlabel("Grade")
plt.ylabel("Delinquency Rate (%)")
plt.show()


This hypothesis checks whether lower grades also show higher delinquency among active loans. This helps connect closed-loan repayment behaviour with current portfolio risk.

The result shows that active-loan delinquency increases from grade A to grade G. Lower-grade active borrowers should therefore be monitored more closely, especially if they are already in grace period or late payment buckets.


## ***Hypothesis 8: Higher interest rates show higher active-loan delinquency***


In [ ]:
active_loans["rate_band"] = pd.cut(
    active_loans["int_rate"],
    bins=[0, 10, 15, 20, 25, 30],
    include_lowest=True
)

rate_total = active_loans.groupby("rate_band", observed=False)["id"].count()

delinquent_active = active_loans[active_loans["loan_status"].isin(delinquent_status)]
rate_delinquent = delinquent_active.groupby("rate_band", observed=False)["id"].count()

rate_delinquency = pd.DataFrame()
rate_delinquency["active_loans"] = rate_total
rate_delinquency["delinquent_loans"] = rate_delinquent
rate_delinquency["delinquent_loans"] = rate_delinquency["delinquent_loans"].fillna(0)
rate_delinquency["delinquency_rate"] = (rate_delinquency["delinquent_loans"] / rate_delinquency["active_loans"]) * 100

display(rate_delinquency["delinquency_rate"].round(2))

plt.figure(figsize=(8, 5))
plt.plot(rate_delinquency.index.astype(str), rate_delinquency["delinquency_rate"], marker="o")
plt.title("Active Loan Delinquency by Interest Rate Band")
plt.xlabel("Interest Rate Band")
plt.ylabel("Delinquency Rate (%)")
plt.xticks(rotation=30)
plt.show()


This hypothesis checks whether higher interest rate loans are also riskier in the active portfolio. This is important because high interest can create pressure before the loan reaches a final status.

The result shows that higher interest bands have higher active delinquency. Borrowers in the highest rate bands should receive earlier reminders or repayment support before the delay becomes serious.


## ***Hypothesis 9: Purpose helps identify active-loan risk***


In [ ]:
purpose_active_total = active_loans.groupby("purpose")["id"].count()

delinquent_active = active_loans[active_loans["loan_status"].isin(delinquent_status)]
purpose_active_delinquent = delinquent_active.groupby("purpose")["id"].count()

purpose_delinquency = pd.DataFrame()
purpose_delinquency["active_loans"] = purpose_active_total
purpose_delinquency["delinquent_loans"] = purpose_active_delinquent
purpose_delinquency["delinquent_loans"] = purpose_delinquency["delinquent_loans"].fillna(0)
purpose_delinquency["delinquency_rate"] = (purpose_delinquency["delinquent_loans"] / purpose_delinquency["active_loans"]) * 100

purpose_delinquency = purpose_delinquency.sort_values("delinquency_rate", ascending=False)
display(purpose_delinquency["delinquency_rate"].round(2))

plt.figure(figsize=(10, 6))
purpose_delinquency["delinquency_rate"].sort_values().plot(kind="barh")
plt.title("Active Loan Delinquency by Purpose")
plt.xlabel("Delinquency Rate (%)")
plt.ylabel("Purpose")
plt.show()


This hypothesis checks whether active delinquency differs by loan purpose. Some purposes may naturally carry more repayment uncertainty than others.

The result shows that some purposes have higher delinquency. However, delinquency rate alone is not enough for prioritisation. It should be read along with outstanding principal, because the bank also needs to know where the money exposure is highest.


## ***Hypothesis 10: Highest current exposure is not always the highest delinquency rate***


In [ ]:
grade_exposure = (
    active_loans
    .groupby("grade")["out_prncp"]
    .sum()
    .sort_values(ascending=False)
)
display(grade_exposure.round(2))

plt.figure(figsize=(8, 5))
plt.bar(grade_exposure.index, grade_exposure.values)
plt.title("Outstanding Principal Exposure by Grade")
plt.xlabel("Grade")
plt.ylabel("Outstanding Principal")
plt.show()


In [ ]:
purpose_exposure = (
    active_loans
    .groupby("purpose")["out_prncp"]
    .sum()
    .sort_values(ascending=False)
)
display(purpose_exposure.head(10).round(2))

plt.figure(figsize=(10, 5))
purpose_exposure.head(10).plot(kind="bar")
plt.title("Top 10 Loan Purposes by Outstanding Exposure")
plt.xlabel("Purpose")
plt.ylabel("Outstanding Principal")
plt.xticks(rotation=45, ha="right")
plt.show()


This section checks where the bank currently has the highest active exposure. A segment can have a lower risk rate but still be important if the outstanding amount is very large.

The result shows that grade C has the highest active outstanding principal, followed by B and D. By purpose, debt consolidation and credit card loans hold the largest exposure. These areas should be watched because even a small increase in delinquency can create a large loss.


## ***At-Risk Outstanding Principal: Where Recovery Should Start***


In [ ]:
at_risk = active_loans[active_loans["loan_status"].isin(delinquent_status)].copy()

risk_grade = (
    at_risk
    .groupby("grade")["out_prncp"]
    .sum()
    .sort_values(ascending=False)
)
display(risk_grade.round(2))

plt.figure(figsize=(8, 5))
risk_grade.plot(kind="bar")
plt.title("At-Risk Outstanding Principal by Grade")
plt.xlabel("Grade")
plt.ylabel("Outstanding Principal")
plt.show()


This section focuses only on active loans that are already in grace period or late status. These loans are the most urgent because they have not fully defaulted yet, but repayment risk has already started.

Grades C, D, and E hold about **INR 167.3M** of at-risk outstanding principal, around **70.9%** of active delinquent exposure. Recovery should start here because this is where the bank has both risk and meaningful money exposure.


# **Key Findings and Recommendations**


## **Key Findings**

1. **Repayment rate falls as grade worsens.** Grades A and B repay better, while grades E, F, and G show weaker repayment.
2. **High interest loans are riskier.** Closed loans with higher interest rates have lower LRR, and active loans with higher rates have higher delinquency.
3. **High DTI borrowers repay worse.** DTI above 30 is linked with weaker repayment performance.
4. **60-month loans perform worse than 36-month loans.** Longer tenure seems to increase repayment uncertainty.
5. **Small business loans need closer monitoring.** They show weaker closed-loan repayment and higher active delinquency.
6. **Exposure is concentrated in grade C/B/D and debt consolidation/credit card loans.** The highest risk rate is not always the highest money exposure.
7. **The strongest recovery starting point is active delinquent loans in grades C, D, and E.** These grades hold the highest at-risk outstanding principal in the current output.


## **Business Recommendations**

### 1. Prioritise active delinquent loans in grades C, D, and E

These grades have the highest at-risk outstanding principal. The bank can create a daily recovery list for borrowers in grace period and late buckets from these grades.

### 2. Monitor risky combinations more closely

Instead of creating a scoring model, the bank can simply watch loans where multiple risk signals appear together, such as lower grade, high interest rate, high DTI, and 60-month term. These combinations showed weaker repayment in the closed-loan analysis.

### 3. Take action based on how late the borrower is

Borrowers in grace period can first receive soft reminders and payment support. Borrowers late by 16-30 days may need calls and payment-date discussion. Borrowers late by 31-120 days should be handled more urgently, with restructuring or settlement options if required.

### 4. Review affordability for high interest loans

Very high interest rate loans show poor repayment behaviour. The bank can improve repayment by checking affordability more carefully and offering support before default.

### 5. Track large exposure purposes separately

Debt consolidation and credit card loans have the highest outstanding exposure. Even a small increase in delinquency here can create a large loss, so these purposes should be monitored separately.


## **Solution to Business Objective**

To improve loan repayment, UCUCI Bank should use both closed-loan learning and active-loan monitoring. The closed-loan analysis shows that grade, interest rate, DTI, term, income, installment, and purpose are useful repayment indicators. The active-loan analysis shows that the biggest immediate recovery opportunity is in delinquent loans from grades C, D, and E.

A simple recovery prioritisation approach is enough for this stage. The bank can first identify borrowers who are already in grace period or late buckets, then rank them by outstanding principal and grade. The loan-age view also helps decide whether the bank should focus more on early repayment behaviour or mid-tenure support.

By acting before loans become charged off, the bank has a better chance of improving repayment rate and reducing losses.


# **Final Conclusion**

UCUCI Bank should focus on two things at the same time:

1. **Improve future lending quality** by being more careful with high-risk loan combinations.
2. **Recover current exposure faster** by prioritising active delinquent loans with high outstanding principal.

The most urgent recovery focus should be on **grades C, D, and E active delinquent loans**, especially when they also have high interest rates, high DTI, long tenure, or risky purposes such as small business.
